In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/playground-series-s5e2/sample_submission.csv
/kaggle/input/playground-series-s5e2/train.csv
/kaggle/input/playground-series-s5e2/test.csv
/kaggle/input/playground-series-s5e2/training_extra.csv


In [2]:
df1 = pd.read_csv("/kaggle/input/playground-series-s5e2/train.csv")
df1.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg),Price
0,0,Jansport,Leather,Medium,7.0,Yes,No,Tote,Black,11.611723,112.15875
1,1,Jansport,Canvas,Small,10.0,Yes,Yes,Messenger,Green,27.078537,68.88056
2,2,Under Armour,Leather,Small,2.0,Yes,No,Messenger,Red,16.643760,39.17320
3,3,Nike,Nylon,Small,8.0,Yes,No,Messenger,Green,12.937220,80.60793
4,4,Adidas,Canvas,Medium,1.0,Yes,Yes,Messenger,Green,17.749338,86.02312


In [3]:
df2 = pd.read_csv("/kaggle/input/playground-series-s5e2/test.csv")
df2.head()

,id,Brand,Material,Size,Compartments,Laptop Compartment,Waterproof,Style,Color,Weight Capacity (kg)
0,300000,Puma,Leather,Small,2.0,No,No,Tote,Green,20.671147
1,300001,Nike,Canvas,Medium,7.0,No,Yes,Backpack,Green,13.564105
2,300002,Adidas,Canvas,Large,9.0,No,Yes,Messenger,Blue,11.809799
3,300003,Adidas,Nylon,Large,1.0,Yes,No,Messenger,Green,18.477036
4,300004,NaN,Nylon,Large,2.0,Yes,Yes,Tote,Black,9.907953


In [4]:
df1.shape, df2.shape

((300000, 11), (200000, 10))

In [5]:
df = pd.concat([df1,df2])

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 500000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   id                    500000 non-null  int64  
 1   Brand                 484068 non-null  object 
 2   Material              486040 non-null  object 
 3   Size                  489024 non-null  object 
 4   Compartments          500000 non-null  float64
 5   Laptop Compartment    487594 non-null  object 
 6   Waterproof            488139 non-null  object 
 7   Style                 486877 non-null  object 
 8   Color                 483265 non-null  object 
 9   Weight Capacity (kg)  499785 non-null  float64
 10  Price                 300000 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 45.8+ MB


In [7]:
df.isnull().sum()

id                           0
Brand                    15932
Material                 13960
Size                     10976
Compartments                 0
Laptop Compartment       12406
Waterproof               11861
Style                    13123
Color                    16735
Weight Capacity (kg)       215
Price                   200000
dtype: int64

In [8]:
df.value_counts("Size")

Size
Medium    169681
Large     164327
Small     155016
Name: count, dtype: int64

In [9]:
s = {"Small": 1, "Medium": 2, "Large": 3}
df["Size"] = df["Size"].map(s)

In [10]:
df.value_counts("Waterproof")

Waterproof
Yes    246671
No     241468
Name: count, dtype: int64

In [11]:
d = {"Yes": 1, "No":0}
df["Waterproof"] = df["Waterproof"].map(d)

In [12]:
df.value_counts("Laptop Compartment")
df["Laptop Compartment"] = df["Laptop Compartment"].map(d)

In [13]:
df["Brand"].fillna(df["Brand"].mode()[0], inplace=True)
df["Material"].fillna(df["Material"].mode()[0], inplace=True)
df["Size"].fillna(df["Size"].mode()[0], inplace=True)
df["Laptop Compartment"].fillna(df["Laptop Compartment"].mode()[0], inplace=True)
df["Waterproof"].fillna(df["Waterproof"].mode()[0], inplace=True)
df["Style"].fillna(df["Style"].mode()[0], inplace=True)
df["Color"].fillna(df["Color"].mode()[0], inplace=True)
df["Weight Capacity (kg)"].fillna(df["Weight Capacity (kg)"].mean(), inplace=True)

In [14]:
df["Brand_Material"] = df["Brand"] + "_" + df["Material"]
df["Style_Size"] = df["Style"] + "_" + df["Size"].astype(str)

df["Weight_per_Compartment"] = df["Weight Capacity (kg)"] / (df["Compartments"] + 1)

In [15]:
df = pd.get_dummies(df, drop_first=True)

In [16]:
train=df[:300000]
test=df[300000:]

In [17]:
x = train.drop(["id","Price"], axis=1)
y = train[["Price"]]
test = test.drop(["id","Price"], axis=1)

In [18]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.15, random_state=8)

In [19]:
xgb_model = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=42)
cat_model = CatBoostRegressor(iterations=300, depth=6, learning_rate=0.05, verbose=0, random_seed=42)
lgb_model = LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42)
rf_model  = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)

xgb_model.fit(x_train, y_train)
cat_model.fit(x_train, y_train)
lgb_model.fit(x_train, y_train)
rf_model.fit(x_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009204 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 611
[LightGBM] [Info] Number of data points in the train set: 255000, number of used features: 47
[LightGBM] [Info] Start training from score 81.388711


RandomForestRegressor(max_depth=10, n_estimators=300, random_state=42)

In [20]:
models = {
    "XGBoost": xgb_model,
    "CatBoost": cat_model,
    "LightGBM": lgb_model,
    "Random Forest": rf_model
}

for name, model in models.items():
    pred = model.predict(x_test)

    rmse = mean_squared_error(y_test, pred, squared=False)
    print(f"RMSE: {rmse:.2f}")

RMSE: 39.03
RMSE: 38.98
RMSE: 38.99
RMSE: 38.99


In [21]:
model = xgb_model.fit(x,y)
prediction = model.predict(test)

In [22]:
prediction

array([85.74666 , 81.48697 , 86.43534 , ..., 83.24988 , 82.13031 ,
       81.431595], dtype=float32)

In [23]:
submission = pd.DataFrame({"id": df2["id"], "Price":prediction})

In [24]:
submission

,id,Price
0,300000,85.746658
1,300001,81.486969
2,300002,86.435341
3,300003,82.448906
4,300004,77.407204
...,...,...
199995,499995,79.778549
199996,499996,85.223114
199997,499997,83.249878
199998,499998,82.130310


In [25]:
submission.to_csv("submission.csv", index=False)